In [22]:
import anthropic
import json
import time
import random
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="C:/Projects/ds-llm-pipeline/.env")
api_key = os.getenv("ANTHROPIC_API_KEY")

client = anthropic.Anthropic(api_key=api_key)

OUTPUT_PATH = "C:/datasets/llm-data/sft"
TARGET_SAMPLES = 100

In [23]:
DS_TOPICS = [
    "linear regression", "overfitting", "confusion matrix",
    "ROC curve", "k means clustering", "PCA",
    "correlation heatmap", "histogram"
]

STATS_TOPICS = [
    "mean and variance", "normal distribution",
    "central limit theorem", "hypothesis testing",
    "confidence intervals", "bayes theorem"
]

MATH_TOPICS = [
    "derivative of x^2", "integration basics",
    "matrix multiplication", "gradient of a function",
    "logarithmic functions"
]

ALL_TOPICS = DS_TOPICS + STATS_TOPICS + MATH_TOPICS

In [24]:
def get_prompt(topic):
    return f"""Generate a data science Q&A pair about "{topic}".

Return ONLY this JSON, nothing else:
{{"instruction": "question about {topic}", "input": "", "output": "### Reasoning:\\nstep by step explanation\\n\\n### Code:\\nimport matplotlib.pyplot as plt\\n# simple working code\\n\\n### Answer:\\nshort conclusion"}}"""

In [25]:
def generate_sample_safe(topic):
    try:
        response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{"role": "user", "content": get_prompt(topic)}])

        text = response.content[0].text.strip()

        #  CLEAN RESPONSE
        text = text.replace("```json", "").replace("```", "").strip()

        #  EXTRACT JSON ONLY
        start = text.find("{")
        end = text.rfind("}") + 1

        if start == -1 or end == -1:
            print("❌ No JSON found")
            return None

        json_text = text[start:end]

        return json.loads(json_text)

    except Exception as e:
        print(" ERROR:", e)
        return None

In [26]:
def validate(sample):
    if not sample:
        return False
    
    if not all(k in sample for k in ["instruction", "input", "output"]):
        return False
    
    out = sample["output"]
    return all(x in out for x in ["### Reasoning:", "### Code:", "### Answer:"])

In [27]:
sample = generate_sample_safe("mean and variance")
print(sample)

{'instruction': 'You have two datasets: Dataset A = [10, 12, 14, 16, 18] and Dataset B = [2, 8, 14, 20, 26]. Both datasets have the same mean of 14. Calculate the variance for both datasets and explain what the difference in variance tells us about the data distribution.', 'input': '', 'output': '### Reasoning:\nTo calculate variance, we need to find the average of the squared differences from the mean. Since both datasets have mean = 14, we calculate:\n- For each value, find (value - mean)²\n- Sum all squared differences\n- Divide by n (population variance) or n-1 (sample variance)\n\nDataset A: All values are close to the mean (differences of ±4, ±2, 0)\nDataset B: Values are more spread out (differences of ±12, ±6, 0)\n\n### Code:\nimport matplotlib.pyplot as plt\nimport numpy as np\n\n# Define datasets\ndataset_A = [10, 12, 14, 16, 18]\ndataset_B = [2, 8, 14, 20, 26]\n\n# Calculate means\nmean_A = np.mean(dataset_A)\nmean_B = np.mean(dataset_B)\n\n# Calculate variances\nvar_A = np.

In [28]:
dataset = []
TARGET = 250
max_attempts = TARGET * 2  # allow retries
attempts = 0

while len(dataset) < TARGET and attempts < max_attempts:
    topic = random.choice(ALL_TOPICS)
    sample = generate_sample_safe(topic)
    attempts += 1

    if validate(sample):
        dataset.append(sample)
        print(f" {len(dataset)}/{TARGET} | {topic}")
    else:
        print(f" Skipped | attempt {attempts}")

    time.sleep(0.5)

print(f"\nDone! {len(dataset)} samples in {attempts} attempts")



 1/250 | central limit theorem
 2/250 | central limit theorem
 3/250 | overfitting
 4/250 | overfitting
 5/250 | histogram
 6/250 | histogram
 7/250 | derivative of x^2
 8/250 | confidence intervals
 9/250 | mean and variance
 10/250 | gradient of a function
 11/250 | linear regression
 12/250 | confidence intervals
 13/250 | PCA
 14/250 | overfitting
 15/250 | hypothesis testing
 16/250 | confusion matrix
 17/250 | matrix multiplication
 18/250 | confusion matrix
 19/250 | linear regression
 20/250 | correlation heatmap
 21/250 | overfitting
 22/250 | derivative of x^2
 23/250 | normal distribution
 24/250 | k means clustering
 25/250 | confidence intervals
 26/250 | logarithmic functions
 27/250 | derivative of x^2
 28/250 | histogram
 29/250 | bayes theorem
 30/250 | k means clustering
 31/250 | mean and variance
 32/250 | PCA
 33/250 | confusion matrix
 34/250 | correlation heatmap
 35/250 | hypothesis testing
 36/250 | ROC curve
 37/250 | histogram
 38/250 | normal distribution
 3

In [29]:
with open("C:/datasets/llm-data/sft/claude_synthetic.json", "w") as f:
    json.dump(dataset, f, indent=2)
print("✓ Saved!")

✓ Saved!
